# 07 — Inference

**Updated:** Uses `mlp_multi_best.pt` (multi-dataset MLP) and the robust ensemble.

Provides:
- `predict_image(path)` — full visual report with anomaly heatmap
- `predict_batch(paths)` — batch CSV output
- SD detection batch
- Usage examples

| Model | Checkpoint | Key metric |
|---|---|---|
| MLP | `mlp_multi_best.pt` | CelebDF AUC 0.9995 · FF++ AUC 0.9968 · SD 100% |
| Ensemble | `ensemble.pkl` (robust) | τ_smooth=0.6387 · smooth arm AUC 0.6573 |

In [1]:
import sys, cv2, numpy as np, pickle
import matplotlib.pyplot as plt
from pathlib import Path
import csv

sys.path.insert(0, '/data/mpstme-naman/deepfake_detection')

BASE     = Path('/data/mpstme-naman/deepfake_detection')
CKPT_DIR = BASE / 'checkpoints'
PROC     = BASE / 'data' / 'processed'
RES_DIR  = BASE / 'results' / 'inference'; RES_DIR.mkdir(parents=True, exist_ok=True)

from config.config_loader          import load_config
from src.features.extractor        import FeatureFusionPipeline
from src.models.mlp_classifier     import MLPTrainer
from src.models.one_class_ensemble import OneClassEnsemble
from src.models.autoencoder        import DeepfakeExplainer

cfg = load_config()

pipeline = FeatureFusionPipeline(cfg=cfg, backbone='clip_vit_b32')
with open(CKPT_DIR / 'pipeline_state.pkl', 'rb') as f:
    pipeline.set_state(pickle.load(f))
print('  ✓  Pipeline loaded')

# PRIMARY MODEL — multi-dataset MLP
mlp = MLPTrainer(cfg=cfg, input_dim=981)
mlp.load_checkpoint(str(CKPT_DIR / 'mlp_multi_best.pt'))
print('  ✓  MLP loaded  (mlp_multi_best.pt)')

# Robust DualOneClassEnsemble
ensemble = OneClassEnsemble(cfg=cfg)
ensemble.load(str(CKPT_DIR / 'ensemble.pkl'))
print(f'  ✓  Ensemble loaded  τ={ensemble.threshold:.4f}  (robust)')

explainer = DeepfakeExplainer(cfg=cfg)
try:
    explainer.load_autoencoder(str(CKPT_DIR / 'autoencoder_best.pt'))
    print('  ✓  Autoencoder loaded')
except Exception as e:
    print(f'  ○  Autoencoder not found — ELA-only mode')

print('\n  All models ready!')
print(f'  MLP : mlp_multi_best.pt  (CelebDF 0.9995 · FF++ 0.9968 · SD 100%)')
print(f'  Ens : ensemble.pkl (robust)  τ_smooth={ensemble.threshold:.4f}')

/usr/local/lib/python3.12/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  CNN Backbone: GPU — NVIDIA H100 PCIe MIG 3g.40gb  (42.4 GB)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 57690.52it/s]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEX

  ✓  Pipeline loaded
  ✓  MLP loaded  (mlp_multi_best.pt)
  DualEnsemble loaded ← /data/mpstme-naman/deepfake_detection/checkpoints/ensemble.pkl
  Ensemble loaded (Dual) ← /data/mpstme-naman/deepfake_detection/checkpoints/ensemble.pkl
  ✓  Ensemble loaded  τ=0.6387  (robust)
  ✓  Autoencoder loaded

  All models ready!
  MLP : mlp_multi_best.pt  (CelebDF 0.9995 · FF++ 0.9968 · SD 100%)
  Ens : ensemble.pkl (robust)  τ_smooth=0.6387


## predict_image()  — Single Image Detection

In [2]:
def predict_image(image_path, save=True):
    """
    Run full deepfake detection on one image.
    Returns a result dict and shows a 3-panel visual report.

    Verdict logic (OR): FAKE if MLP >= 0.5 OR ensemble score > threshold.
    """
    img = cv2.imread(str(image_path))
    if img is None:
        print(f'ERROR: Cannot load {image_path}')
        return None

    Z         = pipeline.extract(img, normalise=True).reshape(1, -1)
    mlp_prob  = float(mlp.predict_proba(Z)[0])
    mlp_label = 'FAKE' if mlp_prob >= 0.5 else 'REAL'
    occ_score = float(ensemble.score(Z)[0])
    occ_label = 'FAKE' if occ_score > ensemble.threshold else 'REAL'
    exp       = explainer.explain(img)

    # Combined verdict: OR logic
    final_label = 'FAKE' if (mlp_label == 'FAKE' or occ_label == 'FAKE') else 'REAL'
    colour      = '#d32f2f' if final_label == 'FAKE' else '#1976d2'

    fig = plt.figure(figsize=(14, 5))
    fig.suptitle(
        f'VERDICT: {final_label}  |  {Path(image_path).name}',
        fontsize=15, fontweight='bold', color=colour
    )
    ax1 = fig.add_subplot(1, 3, 1)
    ax1.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax1.set_title('Input Image'); ax1.axis('off')

    ax2 = fig.add_subplot(1, 3, 2)
    im  = ax2.imshow(exp['heatmap'], cmap='jet', vmin=0, vmax=0.4)
    ax2.set_title('Anomaly Heatmap\n(bright = suspicious region)')
    ax2.axis('off'); plt.colorbar(im, ax=ax2, fraction=0.046)

    ax3 = fig.add_subplot(1, 3, 3)
    ax3.imshow(cv2.cvtColor(exp['overlay'], cv2.COLOR_BGR2RGB))
    ax3.set_title('Overlay'); ax3.axis('off')

    fig.text(
        0.5, -0.03,
        f'MLP (multi-dataset): {mlp_label}  (prob={mlp_prob:.3f})\n'
        f'Ensemble (robust)  : {occ_label}  (score={occ_score:.3f}, τ={ensemble.threshold:.4f})\n'
        f'ELA score          : {exp["score"]:.4f}',
        ha='center', va='top', fontsize=10, family='monospace',
        bbox=dict(boxstyle='round', facecolor='#f5f5f5', alpha=0.9)
    )
    plt.tight_layout()

    if save:
        out = RES_DIR / f'{Path(image_path).stem}_result.png'
        plt.savefig(out, dpi=120, bbox_inches='tight')
    plt.show()

    return {
        'file':        str(image_path),
        'verdict':     final_label,
        'mlp_prob':    round(mlp_prob,  4),
        'mlp_label':   mlp_label,
        'occ_score':   round(occ_score, 4),
        'occ_label':   occ_label,
        'ela_score':   round(exp['score'], 4),
    }

print('  ✓  predict_image() ready')
print('  Usage: result = predict_image("/path/to/image.png")')

  ✓  predict_image() ready
  Usage: result = predict_image("/path/to/image.png")


In [ ]:
# ── CHANGE THIS PATH to any image ────────────────────────────────────────
IMAGE_PATH = PROC / 'celebdf' / 'real' / 'real_0000001.png'
# ─────────────────────────────────────────────────────────────────────────

result = predict_image(IMAGE_PATH)

## predict_batch()  — Batch Detection

In [4]:
def predict_batch(image_paths, save_csv=True):
    """Run detection on a list of image paths. Returns a list of result dicts."""
    results = []
    print(f'Batch inference: {len(image_paths)} images\n')
    for i, path in enumerate(image_paths):
        img = cv2.imread(str(path))
        if img is None:
            continue
        Z      = pipeline.extract(img, normalise=True).reshape(1, -1)
        mp     = float(mlp.predict_proba(Z)[0])
        os_    = float(ensemble.score(Z)[0])
        es     = float(explainer._ela.score(img))
        lbl    = 'FAKE' if (mp >= 0.5 or os_ > ensemble.threshold) else 'REAL'
        print(f'  [{i+1:>4}/{len(image_paths)}]  {Path(path).name:<42}  {lbl}', end='\r')
        results.append({
            'file':      Path(path).name,
            'verdict':   lbl,
            'mlp_prob':  round(mp,   4),
            'occ_score': round(os_,  4),
            'ela_score': round(es,   4),
        })
    print()
    if save_csv and results:
        cp = RES_DIR / 'batch_results.csv'
        with open(cp, 'w', newline='') as f:
            w = csv.DictWriter(f, fieldnames=results[0].keys())
            w.writeheader(); w.writerows(results)
        print(f'  ✓  CSV → {cp}')
    fake_n = sum(1 for r in results if r['verdict'] == 'FAKE')
    print(f'\n  {fake_n}/{len(results)} images detected as FAKE')
    return results

print('  ✓  predict_batch() ready')

  ✓  predict_batch() ready


## Batch Test — 10 real + 10 fake CelebDF

In [5]:
test_paths = (
    sorted((PROC / 'celebdf' / 'real').glob('*.png'))[:10] +
    sorted((PROC / 'celebdf' / 'fake').glob('*.png'))[:10]
)
batch_res = predict_batch(test_paths)

print()
print(f'  {"File":<40} {"Verdict":<8} {"MLP":>7} {"Ensemble":>10}')
print('  ' + '-' * 70)
for r in batch_res:
    print(f'  {r["file"]:<40} {r["verdict"]:<8} '
          f'{r["mlp_prob"]:>7.4f} {r["occ_score"]:>10.4f}')

Batch inference: 20 images

  [  20/20]  fake_0000023.png                            FAKE
  ✓  CSV → /data/mpstme-naman/deepfake_detection/results/inference/batch_results.csv

  10/20 images detected as FAKE

  File                                     Verdict      MLP   Ensemble
  ----------------------------------------------------------------------
  real_0000001.png                         REAL      0.0000     0.4951
  real_0000002.png                         REAL      0.0000     0.4826
  real_0000003.png                         REAL      0.0000     0.5282
  real_0000004.png                         REAL      0.0000     0.5376
  real_0000005.png                         REAL      0.0000     0.5449
  real_0000006.png                         REAL      0.0000     0.4910
  real_0000007.png                         REAL      0.0000     0.4842
  real_0000008.png                         REAL      0.0000     0.5353
  real_0000009.png                         REAL      0.0000     0.5421
  real_0

## Stable Diffusion Detection Batch (30 images)

In [6]:
sd_paths  = sorted((PROC / 'sd' / 'fake').glob('*.png'))[:30]
sd_res    = predict_batch(sd_paths, save_csv=False)
detected  = sum(1 for r in sd_res if r['verdict'] == 'FAKE')
print(f'\n  Stable Diffusion detection: {detected}/{len(sd_res)} = '
      f'{detected/len(sd_res)*100:.1f}%')
print('  (multi-dataset MLP expected: ~100%  |  old single-dataset MLP: 16.7%)')

Batch inference: 30 images

  [  30/30]  fake_0000030.png                            FAKE

  30/30 images detected as FAKE

  Stable Diffusion detection: 30/30 = 100.0%
  (multi-dataset MLP expected: ~100%  |  old single-dataset MLP: 16.7%)


## Usage Examples

In [7]:
# ── Single image ─────────────────────────────────────────────────────────
# result = predict_image('/path/to/image.jpg')

# ── Folder of images ─────────────────────────────────────────────────────
# paths   = list(Path('/your/folder').glob('*.jpg'))
# results = predict_batch(paths)

# ── Check a result dict ──────────────────────────────────────────────────
# print(result['verdict'])    # 'REAL' or 'FAKE'
# print(result['mlp_prob'])   # 0.0 (real) → 1.0 (fake)
# print(result['occ_score'])  # ensemble anomaly score

print('Inference pipeline ready.')
print(f'  Primary model : mlp_multi_best.pt')
print(f'  Ensemble      : ensemble.pkl (robust, τ={ensemble.threshold:.4f})')
print(f'  Output dir    : {RES_DIR}')

Inference pipeline ready.
  Primary model : mlp_multi_best.pt
  Ensemble      : ensemble.pkl (robust, τ=0.6387)
  Output dir    : /data/mpstme-naman/deepfake_detection/results/inference
